# 🇲🇦 Maroc 2017 : situation économique (données FMI)
Notebook Google Colab : tableaux clés + graphiques à partir du fichier Excel **Maroc_2017_donnees_FMI.xlsx**.

## 1. Imports et chargement du fichier Excel

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

plt.rcParams.update({
    "figure.figsize": (10, 5.5), "axes.grid": True, "grid.alpha": 0.3,
    "axes.spines.top": False, "axes.spines.right": False, "font.size": 11,
    "axes.titleweight": "bold", "axes.titlesize": 13,
})
BLEU, ROUGE, VERT, ORANGE, GRIS = "#1F3864", "#C0392B", "#2E8B57", "#E67E22", "#7F8C8D"

# Dans Colab : une fenêtre s'ouvre pour téléverser le fichier Excel
try:
    from google.colab import files
    up = files.upload()
    FICHIER = list(up.keys())[0]
except ImportError:
    FICHIER = "Maroc_2017_donnees_FMI.xlsx"   # exécution hors Colab
print("Fichier utilisé :", FICHIER)

## 2. Lecture des feuilles et préparation des tableaux

In [ ]:
ANNEES = ["2013", "2014", "2015", "2016", "2017", "2018"]
XLAB = ["2013", "2014", "2015", "2016", "2017", "2018\n(proj.)"]

# --- Indicateurs macro 2013-2018 ---
macro = pd.read_excel(FICHIER, sheet_name="Indicateurs macro 2013-2018", header=3).iloc[:, :8]
macro.columns = ["Indicateur", "Unité"] + ANNEES
macro = macro[macro["Unité"].notna()].copy()            # enlève titres de sections et notes
macro["Indicateur"] = macro["Indicateur"].astype(str).str.strip()
for a in ANNEES:
    macro[a] = pd.to_numeric(macro[a], errors="coerce")  # "n.d." -> NaN

def serie(nom, nth=0):
    """Série annuelle (2013-2018) d'un indicateur macro."""
    return macro[macro["Indicateur"] == nom].iloc[nth][ANNEES].astype(float).values

# --- Finances publiques 2017 ---
fp = pd.read_excel(FICHIER, sheet_name="Finances publiques 2017", header=3).iloc[:, :2]
fp.columns = ["Poste", "Mds MAD"]
fp = fp[fp["Poste"].notna()].copy()
fp["Poste"] = fp["Poste"].astype(str).str.strip()
fp["Mds MAD"] = pd.to_numeric(fp["Mds MAD"], errors="coerce")
PIB_MAD = fp.loc[fp["Poste"].str.startswith("PIB nominal"), "Mds MAD"].iloc[0]
fp["% du PIB"] = (fp["Mds MAD"] / PIB_MAD * 100).round(1)

def fpv(debut):
    return fp.loc[fp["Poste"].str.startswith(debut), "Mds MAD"].iloc[0]

# --- Balance des paiements 2017 ---
bp = pd.read_excel(FICHIER, sheet_name="Balance des paiements 2017", header=3).iloc[:, :2]
bp.columns = ["Poste", "Mds USD"]
bp = bp[bp["Poste"].notna()].copy()
bp["Poste"] = bp["Poste"].astype(str).str.strip()
bp["Mds USD"] = pd.to_numeric(bp["Mds USD"], errors="coerce")

def bpv(debut, nth=0):
    return bp.loc[bp["Poste"].str.startswith(debut), "Mds USD"].iloc[nth]

print("Chargement OK :", len(macro), "indicateurs macro,", len(fp), "lignes finances publiques,", len(bp), "lignes balance des paiements")

## 3. Tableau de bord 2017 : les chiffres clés

In [ ]:
cles = [
    "PIB réel", "PIB agricole réel", "PIB non agricole réel",
    "Prix à la consommation (moyenne annuelle)",
    "Solde budgétaire global", "Dette publique totale",
    "Compte courant y c. transferts officiels", "Balance commerciale (marchandises)",
    "Investissements directs étrangers (IDE)", "Réserves brutes de change",
    "Réserves en mois d'importations (année suivante)",
    "Taux de chômage (série du tableau 2 du FMI)", "PIB nominal",
]
tab = macro[macro["Indicateur"].isin(cles)][["Indicateur", "Unité", "2016", "2017"]].copy()
tab["Variation 2017 vs 2016"] = (tab["2017"] - tab["2016"]).round(1)
tab = tab.set_index("Indicateur").loc[cles]
display(tab.style.format({"2016": "{:.1f}", "2017": "{:.1f}", "Variation 2017 vs 2016": "{:+.1f}"})
        .background_gradient(subset=["Variation 2017 vs 2016"], cmap="RdYlGn", vmin=-5, vmax=5))

## 4. Croissance : la dépendance à l'agriculture

In [ ]:
pib, agri, nonagri = serie("PIB réel"), serie("PIB agricole réel"), serie("PIB non agricole réel")
x = np.arange(len(ANNEES))

fig, ax = plt.subplots(1, 2, figsize=(14, 5.5))
ax[0].bar(x, pib, color=BLEU, label="PIB réel")
ax[0].plot(x, nonagri, "o-", color=ORANGE, lw=2.5, label="PIB non agricole")
for i, v in enumerate(pib): ax[0].text(i, v + 0.1, f"{v:.1f}", ha="center", fontsize=9)
ax[0].set_xticks(x); ax[0].set_xticklabels(XLAB)
ax[0].set_ylabel("% de variation annuelle"); ax[0].set_title("Croissance du PIB réel vs hors agriculture")
ax[0].legend()

ax[1].bar(x, agri, color=[VERT if v >= 0 else ROUGE for v in agri])
for i, v in enumerate(agri): ax[1].text(i, v + (0.6 if v >= 0 else -1.8), f"{v:.1f}", ha="center", fontsize=9)
ax[1].axhline(0, color="black", lw=0.8)
ax[1].set_xticks(x); ax[1].set_xticklabels(XLAB)
ax[1].set_title("PIB agricole réel (très volatil)")
plt.tight_layout(); plt.show()

print(f"👉 2017 : PIB +{pib[4]:.1f} % grâce au rebond agricole (+{agri[4]:.1f} %), alors que le hors-agriculture n'avance que de +{nonagri[4]:.1f} %.")
print(f"   En 2016 (sécheresse) : PIB +{pib[3]:.1f} % seulement, agriculture {agri[3]:.1f} %.")

## 5. Croissance vs chômage

In [ ]:
chom = serie("Taux de chômage (série du tableau 2 du FMI)")[:5]   # 2013-2017
x5 = np.arange(5)

fig, ax1 = plt.subplots(figsize=(10, 5.5))
ax1.bar(x5, pib[:5], color=BLEU, alpha=0.75, label="Croissance PIB réel (%)")
ax1.set_ylabel("Croissance du PIB (%)", color=BLEU)
ax1.set_xticks(x5); ax1.set_xticklabels(ANNEES[:5])
ax2 = ax1.twinx()
ax2.plot(x5, chom, "o-", color=ROUGE, lw=2.5, label="Chômage (%)")
for i, v in enumerate(chom): ax2.text(i, v + 0.15, f"{v:.1f}", ha="center", color=ROUGE, fontsize=10)
ax2.set_ylim(8, 11.5); ax2.set_ylabel("Taux de chômage (%)", color=ROUGE); ax2.grid(False)
ax2.spines["right"].set_visible(True)
ax1.set_title("Croissance économique et chômage, 2013-2017")
h1, l1 = ax1.get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels()
ax1.legend(h1 + h2, l1 + l2, loc="upper left")
plt.tight_layout(); plt.show()

print("👉 Le chômage reste autour de 9-10 % malgré le rebond de 2017 : la croissance tirée par l'agriculture crée peu d'emplois durables.")
print("   Complément FMI : chômage 10,6 % au T3 2017 (glissement annuel) et 29,3 % chez les jeunes.")

## 6. Inflation : des prix très maîtrisés

In [ ]:
fin, moy = serie("Prix à la consommation (fin de période)"), serie("Prix à la consommation (moyenne annuelle)")
fig, ax = plt.subplots()
ax.plot(x, moy, "o-", color=BLEU, lw=2.5, label="Moyenne annuelle")
ax.plot(x, fin, "s--", color=ORANGE, lw=2.5, label="Fin de période")
for i in range(len(x)):
    ax.text(i, moy[i] + 0.08, f"{moy[i]:.1f}", ha="center", color=BLEU, fontsize=9)
ax.set_xticks(x); ax.set_xticklabels(XLAB)
ax.set_ylabel("% de variation annuelle"); ax.set_title("Inflation au Maroc")
ax.legend(); plt.show()
print(f"👉 Inflation moyenne 2017 : {moy[4]:.1f} %, bien sous la cible de ~2 % à moyen terme (baisse des prix alimentaires).")

## 7. Finances publiques : déficit et dette

In [ ]:
solde, dette = serie("Solde budgétaire global"), serie("Dette publique totale")
fig, ax1 = plt.subplots(figsize=(10, 5.5))
ax1.bar(x, solde, color=ROUGE, alpha=0.85, label="Solde budgétaire (% PIB)")
for i, v in enumerate(solde): ax1.text(i, v - 0.35, f"{v:.1f}", ha="center", fontsize=9)
ax1.set_ylim(-6.5, 0.5); ax1.set_ylabel("Solde budgétaire (% du PIB)", color=ROUGE)
ax1.set_xticks(x); ax1.set_xticklabels(XLAB)
ax2 = ax1.twinx()
ax2.plot(x, dette, "o-", color=BLEU, lw=2.5, label="Dette publique (% PIB)")
for i, v in enumerate(dette): ax2.text(i, v + 0.35, f"{v:.1f}", ha="center", color=BLEU, fontsize=9)
ax2.set_ylim(55, 70); ax2.set_ylabel("Dette publique (% du PIB)", color=BLEU); ax2.grid(False)
ax2.spines["right"].set_visible(True)
ax1.set_title("Déficit budgétaire en recul, dette autour de 64 % du PIB")
h1, l1 = ax1.get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels()
ax1.legend(h1 + h2, l1 + l2, loc="lower left")
plt.tight_layout(); plt.show()
print(f"👉 Déficit ramené de {solde[0]:.1f} % (2013) à {solde[4]:.1f} % (2017) ; dette stabilisée à {dette[4]:.1f} % du PIB.")

## 8. Structure des recettes et dépenses de l'État en 2017

In [ ]:
rec = {
    "Impôts sur le revenu et bénéfices": fpv("Impôts sur le revenu"),
    "Impôts sur biens et services (TVA…)": fpv("Impôts sur les biens et services"),
    "Impôts sur la propriété": fpv("Impôts sur la propriété"),
    "Impôts sur le commerce international": fpv("Impôts sur le commerce"),
    "Autres impôts": fpv("Autres impôts"),
    "Dons": fpv("Dons"),
    "Autres recettes": fpv("Autres recettes"),
}
dep = {
    "Salaires de la fonction publique": fpv("Rémunération des salariés"),
    "Biens, services et transferts": fpv("Utilisation de biens et services"),
    "Intérêts de la dette": fpv("Intérêts de la dette"),
    "Subventions": fpv("Subventions"),
    "Autres charges": fpv("Autres charges"),
    "Investissement": fpv("Acquisition nette"),
}
fig, ax = plt.subplots(1, 2, figsize=(15, 6))
for a, d, titre, pal in [(ax[0], rec, f"Recettes 2017 : {fpv('Recettes'):.0f} Mds MAD", plt.cm.Blues),
                         (ax[1], dep, f"Dépenses 2017 : charges {fpv('Dépenses'):.0f} + investissement {fpv('Acquisition nette'):.0f} Mds MAD", plt.cm.Oranges)]:
    vals = np.array(list(d.values()))
    cols = pal(np.linspace(0.85, 0.3, len(vals)))
    w, _, autot = a.pie(vals, colors=cols, startangle=90, counterclock=False, autopct="%1.0f%%",
                        pctdistance=0.8, wedgeprops=dict(width=0.45, edgecolor="white"))
    a.legend(w, [f"{k} ({v:.0f})" for k, v in d.items()], loc="upper center", bbox_to_anchor=(0.5, 0.02), fontsize=9, ncol=2)
    a.set_title(titre); a.grid(False)
plt.tight_layout(); plt.show()

print(f"👉 Les salaires publics absorbent {dep['Salaires de la fonction publique']/fpv('Dépenses')*100:.0f} % des dépenses courantes ; "
      f"les intérêts de la dette : {dep['Intérêts de la dette']:.1f} Mds MAD.")
print(f"   Le déficit global 2017 : {fpv('Solde global'):.1f} Mds MAD, soit {fpv('Solde global')/PIB_MAD*100:.1f} % du PIB.")

## 9. Secteur extérieur : un déficit commercial structurel

In [ ]:
ca, comm = serie("Compte courant y c. transferts officiels"), serie("Balance commerciale (marchandises)")
w = 0.38
fig, ax = plt.subplots()
ax.bar(x - w/2, comm, w, color=ROUGE, label="Balance commerciale")
ax.bar(x + w/2, ca, w, color=BLEU, label="Compte courant")
for i in range(len(x)):
    ax.text(i - w/2, comm[i] - 1.2, f"{comm[i]:.1f}", ha="center", fontsize=8)
    ax.text(i + w/2, ca[i] - 1.2, f"{ca[i]:.1f}", ha="center", fontsize=8)
ax.set_xticks(x); ax.set_xticklabels(XLAB); ax.set_ylim(-24, 1)
ax.set_ylabel("% du PIB"); ax.set_title("Déficit commercial et compte courant")
ax.legend(loc="lower right"); plt.show()

# Cascade du compte courant 2017
etapes = [("Balance\ncommerciale", bpv("Balance commerciale")), ("Services\n(tourisme…)", bpv("Services")),
          ("Revenus", bpv("Revenus")), ("Transferts\n(MRE, dons)", bpv("Transferts courants"))]
fig, ax = plt.subplots()
cumul = 0
for i, (nom, v) in enumerate(etapes):
    ax.bar(i, v, bottom=cumul if v >= 0 else cumul, color=VERT if v >= 0 else ROUGE)
    ax.text(i, cumul + v / 2, f"{v:+.1f}", ha="center", va="center", color="white", fontweight="bold")
    cumul += v
ax.bar(len(etapes), cumul, color=BLEU); ax.text(len(etapes), cumul / 2, f"{cumul:.1f}", ha="center", va="center", color="white", fontweight="bold")
ax.set_xticks(range(len(etapes) + 1)); ax.set_xticklabels([e[0] for e in etapes] + ["Compte\ncourant"])
ax.set_ylim(-20, 1); ax.axhline(0, color="black", lw=0.8); ax.set_ylabel("Milliards USD"); ax.set_title("Comment se forme le compte courant en 2017")
plt.show()
print("👉 Le Maroc importe bien plus qu'il n'exporte (-18 Mds USD) ; tourisme et envois des MRE comblent l'essentiel du trou.")

## 10. Ce que le Maroc exporte et importe (2017)

In [ ]:
exp = {"Automobiles": bpv("dont automobiles"), "Produits alimentaires": bpv("dont produits alimentaires", 0), "Phosphates et dérivés": bpv("dont phosphates")}
imp = {"Biens d'équipement": -bpv("dont biens d'équipement"), "Énergie": -bpv("dont énergie"), "Produits alimentaires": -bpv("dont produits alimentaires", 1)}

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
for a, d, col, t in [(ax[0], exp, VERT, "Principales exportations (Mds USD)"), (ax[1], imp, ROUGE, "Principales importations (Mds USD)")]:
    ks = list(d.keys()); vs = list(d.values())
    a.barh(ks, vs, color=col)
    for i, v in enumerate(vs): a.text(v + 0.1, i, f"{v:.1f}", va="center")
    a.set_title(t); a.invert_yaxis()
plt.tight_layout(); plt.show()
print("👉 Les automobiles sont devenues le 1er poste d'exportation, devant les phosphates et l'alimentaire.")

## 11. Réserves de change

In [ ]:
res, mois = serie("Réserves brutes de change"), serie("Réserves en mois d'importations (année suivante)")
fig, ax1 = plt.subplots(figsize=(10, 5.5))
ax1.bar(x, res, color=BLEU, alpha=0.85, label="Réserves (Mds USD)")
for i, v in enumerate(res): ax1.text(i, v + 0.4, f"{v:.1f}", ha="center", fontsize=9)
ax1.set_xticks(x); ax1.set_xticklabels(XLAB); ax1.set_ylabel("Milliards USD")
ax2 = ax1.twinx(); ax2.plot(x, mois, "o-", color=ORANGE, lw=2.5, label="Mois d'importations")
ax2.set_ylim(0, 8); ax2.set_ylabel("Mois d'importations"); ax2.grid(False); ax2.spines["right"].set_visible(True)
ax1.set_title("Réserves de change : environ 6 mois d'importations")
h1, l1 = ax1.get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels()
ax1.legend(h1 + h2, l1 + l2, loc="upper left")
plt.tight_layout(); plt.show()
print(f"👉 Fin 2017 : {res[4]:.1f} Mds USD, soit {mois[4]:.1f} mois d'importations (léger recul par rapport à 2016).")

## 12. Synthèse

In [ ]:
print("SITUATION DU MAROC EN 2017 (estimations FMI)")
print("-" * 60)
print(f"• Croissance : +{pib[4]:.1f} % (rebond agricole), hors agriculture : +{nonagri[4]:.1f} %")
print(f"• Inflation moyenne : {moy[4]:.1f} %")
print(f"• Déficit budgétaire : {solde[4]:.1f} % du PIB | Dette : {dette[4]:.1f} % du PIB")
print(f"• Compte courant : {ca[4]:.1f} % du PIB | Déficit commercial : {comm[4]:.1f} % du PIB")
print(f"• Réserves : {res[4]:.1f} Mds USD ({mois[4]:.1f} mois d'importations)")
print(f"• Chômage : {chom[4]:.1f} % (10,6 % au T3 ; 29,3 % chez les jeunes)")
print("\nPoints forts : stabilité macro (inflation, réserves), déficit en baisse, exportations industrielles en hausse.")
print("Points faibles : croissance dépendante de la pluie, chômage élevé surtout chez les jeunes, déficit commercial.")